# 01 — Génération des métriques · Nowledgeable

Ce notebook exécute **uniquement** les indicateurs déclarés dans `metric_registry.py`
pour le corpus **Nowledgeable**. Les sorties sont isolées dans `csv/Nowledgeable/` et les
journaux dans `csv/Nowledgeable/logs/`.

Tous les scripts sont lancés dans des sous-processus indépendants afin d'éviter les
effets de bord entre imports, loggers et arguments de ligne de commande.


## 1. Configuration


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, inspect_input, run_metric_jobs, metric_inventory

PROJECT_DIR = detect_project_dir()
DATASET = "Nowledgeable"
SPEC = get_dataset(DATASET)

# Laisser à None pour utiliser le chemin défini dans metric_registry.py.
# Exemple Mirabelle : PROJECT_DIR / "data" / "autres_traces.csv"
# Exemple ProgSnap2 : PROJECT_DIR / "data"  (dossier contenant MainTable.csv)
INPUT_OVERRIDE = None

OVERWRITE_OUTPUTS = True

print("Projet       :", PROJECT_DIR)
print("Corpus       :", DATASET)
print("Scripts      :", SPEC.scripts_path(PROJECT_DIR))
print("Sorties CSV  :", SPEC.csv_dir(PROJECT_DIR))
print("Entrée défaut:", SPEC.input(PROJECT_DIR))


## 2. Contrôle de l'entrée


In [ ]:
input_df, input_summary = inspect_input(PROJECT_DIR, SPEC, INPUT_OVERRIDE)
display(input_summary)
print("Colonnes disponibles :")
print(", ".join(input_df.columns.astype(str)))


## 3. Indicateurs déclarés


In [ ]:
jobs_df = pd.DataFrame([
    {
        "script": item.script,
        "sortie": item.output,
        "métrique": item.metric,
        "description": item.description,
        "colonnes requises": ", ".join(item.required_columns),
        "arguments": " ".join(item.extra_args),
    }
    for item in SPEC.jobs
])
display(jobs_df)


## 4. Exécution


In [ ]:
report_df = run_metric_jobs(
    PROJECT_DIR,
    DATASET,
    overwrite=OVERWRITE_OUTPUTS,
    input_override=INPUT_OVERRIDE,
)
display(report_df[[
    "script", "metric", "status", "rows", "columns", "message", "log_path"
]])

errors = report_df[~report_df["status"].isin(["ok", "skipped_existing"])]
if errors.empty:
    print("Tous les indicateurs ont été générés correctement.")
else:
    print(f"{len(errors)} indicateur(s) à vérifier. Consultez les fichiers log_path.")


## 5. Inventaire des sorties


In [ ]:
inventory_df = metric_inventory(SPEC.csv_dir(PROJECT_DIR))
display(inventory_df)
